# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook demonstrates a full workflow for accessing, exploring, and preparing the FAIR^2 Colorectal Cancer dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

You will use this schema to load record sets, fields, and tabular data—all addressed by their `@id` properties as per the Croissant standard.

In [ ]:
# Install and import mlcroissant (uncomment if not already installed)
!pip install -U mlcroissant

## 1. Data Loading
First, load metadata and records from the dataset using `mlcroissant`. The main entities to explore (record sets, fields, etc.) will become available once the metadata is loaded.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL for this dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset via Croissant
dataset = mlc.Dataset(croissant_url)

# Access main metadata fields (see README for full attributes)
md = dataset.metadata
print(f"Dataset name: {md.name}\n")
print(f"Short description: {md.description}\n")
print(f"Number of record sets: {len(md.record_sets)}")

## 2. Data Overview

Review the available record sets, and for each, print its `@id`, name, and a summary of available fields (with their respective `@id`s and types). This will help identify which @ids to use in downstream analyses.

**Note**: As per Croissant schemas, record sets and fields are always referenced by `@id`.

In [ ]:
# Iterate through all available record sets and list their details
record_sets = md.record_sets

if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    for rs in record_sets:
        print(f'Record set @id: {rs["@id"]}')
        print(f'  Name: {getattr(rs, "name", "N/A")}')
        print(f'  Description: {getattr(rs, "description", "N/A")}')
        fields = getattr(rs, "fields", [])
        print(f'  Fields:')
        for field in fields:
            # Each field is an object with its own @id
            print(f'    - @id: {field["@id"]} (name: {getattr(field, "name", "N/A")}, type: {getattr(field, "data_type", "N/A")})')
        print('\n')
# Keep the first record set's @id for further extraction
if record_sets:
    main_record_set_id = record_sets[0]["@id"]

## 3. Data Extraction

Load tabular data for record sets of interest. For demonstration, this notebook will extract all available record sets and load them into `pandas` DataFrames, addressed by their `@id` properties.

In [ ]:
# List all record set @ids
record_set_ids = [rs["@id"] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    # Keep only non-empty DataFrames
    if not df.empty:
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print("  No records found.")

if dataframes:
    # For example, show columns and head of the first non-empty dataframe
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns for record set {first_rs_id}: {dataframes[first_rs_id].columns.tolist()}")
    dataframes[first_rs_id].head()
else:
    print("No dataframes were loaded. Check the record sets for data.")

## 4. Exploratory Data Analysis (EDA)

We'll apply typical data analysis steps. For demonstration, choose one record set (the first non-empty), then pick a numeric field (by inspecting the DataFrame) and a grouping field (categorical or discrete.) All fields are referenced by their Croissant `@id` (column names in the DataFrame).

Operations below include filtering on some threshold, normalizing numeric data, and aggregating by a grouping key.

In [ ]:
import numpy as np

if dataframes:
    record_set_id = first_rs_id
    df = dataframes[record_set_id]
    print(f"Working with DataFrame for record set: {record_set_id}")
    print(df.head())

    # Attempt to identify numeric fields and a candidate group field
    numeric_field_candidates = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    if not numeric_field_candidates:
        print("No numeric columns detected; attempting to convert potential numeric columns.")
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors="ignore")
            except Exception:
                pass
        numeric_field_candidates = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]

    if numeric_field_candidates:
        # Use the first numeric field
        numeric_field = numeric_field_candidates[0]
        print(f"Selected numeric field by @id: {numeric_field}")

        # Example: filter for values greater than a threshold (10)
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to select a grouping field (categorical, with < N distinct values)
        candidate_group_fields = [col for col in df.columns if df[col].nunique() < 10 and col != numeric_field]
        if candidate_group_fields:
            group_field = candidate_group_fields[0]
            print(f"\nGrouping by field @id: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df)
        else:
            print("No suitable group field found.")
    else:
        print("No numeric columns available for EDA.")
else:
    print("No available data for EDA.")

## 5. Visualization

Let's visualize the distribution of the selected numeric variable (e.g., a histogram), and if available, the grouped means. This step helps reveal patterns or outliers in the data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_candidates:
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))

    # Plot the original numeric field
    sns.histplot(df[numeric_field], kde=True, ax=axs[0], color="skyblue")
    axs[0].set_title(f"Distribution of {numeric_field}")
    axs[0].set_xlabel(numeric_field)

    # Plot normalized values from filtered_df, if it exists and enough data
    if 'filtered_df' in locals() and not filtered_df.empty:
        sns.histplot(filtered_df[f"{numeric_field}_normalized"], kde=True, ax=axs[1], color="salmon")
        axs[1].set_title(f"Normalized {numeric_field} (Filtered)")
        axs[1].set_xlabel(f"{numeric_field}_normalized")

    plt.tight_layout()
    plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion

In this notebook, you have:

- Loaded the FAIR^2 Clinicopathological Colorectal Cancer dataset via its Croissant schema
- Reviewed record sets, fields, and their unique `@id`s
- Extracted tabular data using the `mlcroissant` library
- Performed filtering, normalization, and grouping on a selected numeric property
- Visualized the distribution and summary statistics for deeper understanding

**Next steps:** This framework allows for deeper clinical exploration of cancer progression, anatomical trends, or MSI-H status analysis using the field and record set `@id`s revealed above.
<br><br>
*For more details and advanced analytics, see the [mlcroissant documentation](https://mlcroissant.readthedocs.io) and Croissant schema guides.*